# Extract Latent Vectors from Trained VAE

This notebook extracts latent space vectors from a trained VAE to use as input for the MDN-RNN.

**Input**: Trained VAE checkpoint + KITTI images  
**Output**: Latent sequences saved to `data/latent_sequences.pt`

In [1]:
import sys
sys.path.insert(0, '..')  # Add parent directory to path

import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torchvision.transforms as transforms

from src.models.world_model import ConvVAE
from src.data.loaders import KITTIDataset

if torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")

Using device: cpu


## 1. Load Trained VAE

In [2]:
# Specify checkpoint path
checkpoint_path = '../outputs/vae_z64_img128_rect/checkpoints/vae_final.pth'

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)
cfg = checkpoint['config']

print("Configuration:")
print(f"  Latent dim: {cfg['model']['vae']['latent_dim']}")
print(f"  Image size: {cfg['data'].get('img_height', cfg['data'].get('img_size'))}x{cfg['data'].get('img_width', cfg['data'].get('img_size'))}")
print(f"  Final PSNR: {checkpoint['loss_history']['psnr'][-1]:.2f} dB")

Configuration:
  Latent dim: 64
  Image size: 128x416
  Final PSNR: 20.92 dB


In [3]:
# Initialize model
latent_dim = cfg['model']['vae']['latent_dim']

# Handle both square and rectangular configs
if 'img_size' in cfg['data']:
    img_height = cfg['data']['img_size']
    img_width = cfg['data']['img_size']
else:
    img_height = cfg['data']['img_height']
    img_width = cfg['data']['img_width']

model = ConvVAE(
    latent_dim=latent_dim,
    img_height=img_height,
    img_width=img_width
)

# Load weights
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print(f"\nModel loaded successfully!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


Model loaded successfully!
Parameters: 11,657,027


## 2. Load Dataset with Sequences

In [4]:
# Configuration for sequence extraction
SEQUENCE_LENGTH = 20  # Number of frames per sequence for RNN
DATA_PATH = cfg['data']['path']

# Create transform
transform = transforms.Compose([
    transforms.Resize((img_height, img_width)),
    transforms.ToTensor()
])

# Load dataset
dataset = KITTIDataset(
    root_dir=f'../{DATA_PATH}',
    sequence_length=SEQUENCE_LENGTH,
    transform=transform
)

print(f"Dataset loaded: {len(dataset)} sequences")
print(f"Sequence length: {SEQUENCE_LENGTH} frames")
print(f"Total frames: {len(dataset) + SEQUENCE_LENGTH - 1}")

Dataset loaded: 4525 sequences
Sequence length: 20 frames
Total frames: 4544


## 3. Extract Latent Vectors

In [ ]:
# Extract latent vectors for all sequences
latent_sequences = []

with torch.no_grad():
    for i in tqdm(range(len(dataset)), desc="Extracting latents"):
        # Get sequence: (seq_len, 3, H, W)
        seq = dataset[i].to(device)
        
        # Encode each frame to latent space
        z_seq = model.encode(seq)  # (seq_len, latent_dim)
        
        latent_sequences.append(z_seq.cpu())

# Stack all sequences: (num_sequences, seq_len, latent_dim)
latents = torch.stack(latent_sequences)

print(f"\nLatent data shape: {latents.shape}")
print(f"  - {latents.shape[0]} sequences")
print(f"  - {latents.shape[1]} frames per sequence")
print(f"  - {latents.shape[2]} latent dimensions")

Extracting latents:  70%|███████   | 3169/4525 [42:20<20:16,  1.11it/s] 

## 4. Visualize Latent Statistics

In [ ]:
# Compute statistics
latents_flat = latents.view(-1, latent_dim)  # (num_sequences * seq_len, latent_dim)

mean = latents_flat.mean(dim=0).numpy()
std = latents_flat.std(dim=0).numpy()

print(f"Latent statistics:")
print(f"  Mean range: [{mean.min():.3f}, {mean.max():.3f}]")
print(f"  Std range: [{std.min():.3f}, {std.max():.3f}]")
print(f"  Overall mean: {latents_flat.mean():.3f}")
print(f"  Overall std: {latents_flat.std():.3f}")

In [ ]:
# Plot latent dimension statistics
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Mean per dimension
axes[0].bar(range(latent_dim), mean, alpha=0.7)
axes[0].set_xlabel('Latent Dimension')
axes[0].set_ylabel('Mean')
axes[0].set_title('Mean per Latent Dimension')
axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.5)

# Std per dimension
axes[1].bar(range(latent_dim), std, alpha=0.7, color='orange')
axes[1].set_xlabel('Latent Dimension')
axes[1].set_ylabel('Standard Deviation')
axes[1].set_title('Std Dev per Latent Dimension')
axes[1].axhline(y=1, color='r', linestyle='--', alpha=0.5, label='Target (1.0)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Plot histogram of latent values
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(latents_flat.numpy().flatten(), bins=100, alpha=0.7, density=True)
ax.set_xlabel('Latent Value')
ax.set_ylabel('Density')
ax.set_title('Distribution of Latent Values')

# Overlay standard normal for comparison
x = np.linspace(-4, 4, 100)
ax.plot(x, np.exp(-x**2/2) / np.sqrt(2*np.pi), 'r--', label='N(0,1)')
ax.legend()

plt.show()

## 5. Visualize Sample Sequence in Latent Space

In [ ]:
# Plot first 3 latent dimensions over time for a sample sequence
sample_idx = 0
sample_seq = latents[sample_idx].numpy()  # (seq_len, latent_dim)

fig, ax = plt.subplots(figsize=(12, 4))

for dim in range(min(5, latent_dim)):
    ax.plot(sample_seq[:, dim], label=f'z_{dim}', alpha=0.7)

ax.set_xlabel('Time Step')
ax.set_ylabel('Latent Value')
ax.set_title(f'Latent Dimensions Over Time (Sequence {sample_idx})')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.show()

## 6. Save Latent Data for RNN Training

In [ ]:
# Save latent sequences
import os

save_path = '../data/latent_sequences.pt'
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# Save with metadata
save_data = {
    'latents': latents,
    'latent_dim': latent_dim,
    'sequence_length': SEQUENCE_LENGTH,
    'num_sequences': len(latents),
    'vae_checkpoint': checkpoint_path,
    'config': cfg
}

torch.save(save_data, save_path)

print(f"Saved latent data to: {save_path}")
print(f"File size: {os.path.getsize(save_path) / 1024 / 1024:.2f} MB")

## 7. Verify Reconstruction Quality

In [ ]:
# Test reconstruction from latent vectors
sample_seq = dataset[0].to(device)  # (seq_len, 3, H, W)

with torch.no_grad():
    # Encode
    z = model.encode(sample_seq)
    
    # Decode
    recon = model.decode(z)

# Plot original vs reconstruction
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i in range(5):
    # Original
    img_orig = sample_seq[i*4].cpu().permute(1, 2, 0).numpy()
    axes[0, i].imshow(img_orig)
    axes[0, i].set_title(f'Original t={i*4}')
    axes[0, i].axis('off')
    
    # Reconstruction
    img_recon = recon[i*4].cpu().permute(1, 2, 0).numpy()
    axes[1, i].imshow(img_recon)
    axes[1, i].set_title(f'Reconstructed')
    axes[1, i].axis('off')

plt.suptitle('VAE Encode → Decode Verification', fontsize=14)
plt.tight_layout()
plt.show()

## Usage in MDN-RNN

Load the saved latent data for RNN training:

```python
# Load latent sequences
data = torch.load('data/latent_sequences.pt')
latents = data['latents']  # (num_sequences, seq_len, latent_dim)

# For RNN training:
# Input: latents[:, :-1, :]   # z_1 to z_{t-1}
# Target: latents[:, 1:, :]   # z_2 to z_t
```